# 01 Region Setup

This notebook prepares the Austin inland flood region. It uses the merged Region config, recognizes the SMART-DS evaluation footprint, selects the SFINCS coverage named in `sfincs_domain_set.include_domain_ids`, wraps that coverage in the encompassing Wflow HUC watershed, collects the static inputs required by Wflow and SFINCS, and plots the collected products.

## Inland Adaptation: Selected Coverage Inside a Watershed

SFINCS owns the hydraulic coverage around the selected SMART-DS subregion. Wflow owns the larger HUC watershed that drains into that selected coverage. The only notebook-facing domain choice is the Location override entry in `locations/austin/domain.yaml`.

**Methodological Stages:**

### 1. Parameters and Configuration

Load the Austin Region config and show the path syntax used for local records.

### 2. Footprint and Watershed

Build/read the SMART-DS evaluation footprint, write the selected SFINCS coverage box, fetch/select the enclosing WBD HUC watershed, and plot the relationship.

### 3. Required Static Data

Collect Wflow static inputs first over the larger HUC-derived boundary, then collect SFINCS static inputs over the selected hydraulic coverage.

### 4. Collected Input Plot

Plot the DEM, landcover, hydrologic soil group, and saturated-conductivity rasters for both model domains.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

location_root = Path.cwd().parent
repo_root = location_root.parents[1]
src_root = repo_root / "src"
src_path = str(src_root)
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
importlib.invalidate_caches()
for module_name in [name for name in sys.modules if name == "sfincs_runs" or name.startswith("sfincs_runs.")]:
    del sys.modules[module_name]

import sfincs_runs.build_base.region_notebook as region

pd.set_option("display.max_colwidth", 140)


## 1. Parameters & Configuration


In [ ]:
runtime = region.load_region_setup_notebook_runtime(location_root)

pd.Series(
    {
        "location": runtime.config["project"]["place_name"],
        "selected_sfincs_domain_ids": runtime.sfincs_config["sfincs_domain_set"]["include_domain_ids"],
        "source_record_syntax": 'location_root / "data" / "sources" / "..."',
        "static_record_syntax": 'location_root / "data" / "static" / "..."',
    },
    name="region_setup_parameters",
)


## 2. SMART-DS Footprint and Wflow HUC Watershed


In [ ]:
domains = region.build_selected_inland_region_domains(runtime)

display(domains.footprint.summary)
display(domains.summary)
region.plot_selected_inland_region(runtime, domains)
plt.show()


## 3. Required Wflow and SFINCS Static Data


In [ ]:
static_data = region.collect_required_inland_static_data(runtime)

display(static_data.wflow_summary)
static_data.sfincs_summary


## 4. Collected Input Plot


In [ ]:
static_plot = region.plot_collected_inland_static_data(runtime)
if isinstance(static_plot, pd.DataFrame):
    display(static_plot)
else:
    plt.show()
